In [10]:
import cv2
import numpy as np
from scipy import ndimage
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse

In [11]:
#phương pháp bicubic
img = cv2.imread("15.jpg")

# resize bằng bicubic
bicubic = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

img1 = bicubic
cv2.imwrite("15_bicubic.jpg", img1)

True

In [12]:
# EIAAF
gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)

# tính gradient
gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)

mag = cv2.magnitude(gx, gy)
dir = cv2.phase(gx, gy)

# mờ theo hướng cạnh
blur1 = cv2.GaussianBlur(img1, (5, 5), 0.5)
mask = cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX)

# trộn
mask = np.stack([mask] * 3, axis=-1)
jaggy_suppressed = img1*(1-mask) + blur1*mask
jaggy_suppressed = jaggy_suppressed.astype(np.uint8)
img2 = jaggy_suppressed
cv2.imwrite("15_EIAAF.jpg", img2)

True

In [13]:
# EDUMS
def tinh_gradient(anh):
    sobel_x = np.array([[-1, 0, 1],
                        [-2, 0, 2],
                        [-1, 0, 1]])
    sobel_y = np.array([[-1, -2, -1],
                        [ 0,  0,  0],
                        [ 1,  2,  1]])
    gx = ndimage.convolve(anh, sobel_x, mode='reflect')
    gy = ndimage.convolve(anh, sobel_y, mode='reflect')
    do_lon = np.sqrt(gx**2 + gy**2)
    huong = np.arctan2(gy, gx)
    return do_lon, huong, gx, gy

def rgb_to_ycbcr_fullrange(rgb):
    # rgb: float in 0..255
    R = rgb[...,0]; G = rgb[...,1]; B = rgb[...,2]
    Y  =  0.29900 * R + 0.58700 * G + 0.11400 * B
    Cb = 128.0 + (-0.168736 * R - 0.331264 * G + 0.5 * B)
    Cr = 128.0 + (0.5 * R - 0.418688 * G - 0.081312 * B)
    return Y, Cb, Cr

def ycbcr_to_rgb_fullrange(Y, Cb, Cr):
    cb = Cb - 128.0
    cr = Cr - 128.0
    R = Y + 1.40200 * cr
    G = Y - 0.344136 * cb - 0.714136 * cr
    B = Y + 1.77200 * cb
    return np.stack([R, G, B], axis=-1)

def edums_nhanh_ycbcr(anh, alpha=1.5, sigma=2.0, bgr=False, verbose=False):
    """
    Sharpen chỉ trên kênh Y (YCbCr full-range), giữ Cb/Cr nguyên vẹn.
    Trả về ảnh cùng dtype/scale như input. Nếu bgr=True thì input/output là BGR (cv2).
    Nếu verbose=True: in thông tin debug.
    """
    orig_dtype = anh.dtype
    arr = anh.astype(np.float64)

    # detect float 0..1 scaling
    scaled_from_0_1 = False
    if np.issubdtype(orig_dtype, np.floating) and arr.max() <= 1.0 + 1e-12:
        arr = arr * 255.0
        scaled_from_0_1 = True

    has_alpha = (arr.ndim == 3 and arr.shape[2] == 4)
    alpha_chan = None
    if has_alpha:
        alpha_chan = arr[...,3].copy()
        arr = arr[...,:3]

    if verbose:
        print("DEBUG: arr.shape", arr.shape, "dtype", orig_dtype, "scaled_from_0_1", scaled_from_0_1)

    # If grayscale input (HxW)
    if arr.ndim == 2:
        Y = arr
        do_lon, huong, gx, gy = tinh_gradient(Y)
        max_m = np.max(do_lon)
        do_lon_norm = do_lon / (max_m + 1e-10)
        w = np.clip(do_lon_norm, 0.0, 1.0)
        Y_blur = gaussian_filter(Y, sigma=sigma)
        mask = Y - Y_blur
        Y_sharp = np.clip(Y + alpha * mask * w, 0.0, 255.0)
        out = Y_sharp
    else:
        rgb = arr.copy()
        if bgr:
            rgb = rgb[..., ::-1]  # BGR -> RGB for processing

        # Convert to YCbCr (works with 0..255 float)
        Y, Cb, Cr = rgb_to_ycbcr_fullrange(rgb)

        # compute edge weight on Y
        do_lon, huong, gx, gy = tinh_gradient(Y)
        max_m = np.max(do_lon)
        do_lon_norm = do_lon / (max_m + 1e-10)
        w = np.clip(do_lon_norm, 0.0, 1.0)

        Y_blur = gaussian_filter(Y, sigma=sigma)
        mask = Y - Y_blur
        Y_sharp = np.clip(Y + alpha * mask * w, 0.0, 255.0)

        # Reconstruct RGB from sharpened Y and original Cb/Cr
        out_rgb = ycbcr_to_rgb_fullrange(Y_sharp, Cb, Cr)
        # Clip and ensure shape HxWx3
        out_rgb = np.clip(out_rgb, 0.0, 255.0)

        if bgr:
            out_rgb = out_rgb[..., ::-1]

        out = out_rgb

    # reattach alpha channel if present
    if has_alpha:
        alpha_chan_clipped = np.clip(alpha_chan, 0.0, 255.0)
        out = np.concatenate([out, alpha_chan_clipped[..., np.newaxis]], axis=2)

    # cast back to original dtype/scale
    if scaled_from_0_1:
        out = out / 255.0
        out = out.astype(orig_dtype)
    else:
        if np.issubdtype(orig_dtype, np.integer):
            out = np.clip(out, 0, 255).round().astype(orig_dtype)
        else:
            out = out.astype(orig_dtype)

    if verbose:
        print("DEBUG: out.shape", out.shape, "out.dtype", out.dtype,
              "min/max:", out.min(), out.max())
        if out.ndim == 3:
            print("DEBUG: channel means:", [out[...,c].mean() for c in range(out.shape[2])])

    return out

In [14]:
# in ra ảnh sau khi dùng EDUMS
img3 = edums_nhanh_ycbcr(img2, alpha=1.5, sigma=1.5, bgr=True, verbose=True)
cv2.imwrite('15_edums.png', img3)

DEBUG: arr.shape (1600, 2400, 3) dtype uint8 scaled_from_0_1 False
DEBUG: out.shape (1600, 2400, 3) out.dtype uint8 min/max: 0 255
DEBUG: channel means: [41.48845416666666, 117.48134973958334, 117.63312734375]


True

In [15]:
#Khử rung EICL
def phat_hien_canh_sobel(anh_kenh, scale=1.0):
    """
    Phát hiện cạnh bằng Sobel trên một kênh
    """
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]) * scale
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]) * scale
    
    gx = ndimage.convolve(anh_kenh, sobel_x, mode='reflect')
    gy = ndimage.convolve(anh_kenh, sobel_y, mode='reflect')
    
    edge_magnitude = np.sqrt(gx**2 + gy**2)
    edge_magnitude = edge_magnitude / (edge_magnitude.max() + 1e-10)
    
    return edge_magnitude


def tao_ban_do_trong_so(edge_map, nguong=0.1, sigma_expand=2.0):
    """
    Tạo bản đồ trọng số từ edge map
    Vùng gần cạnh có trọng số cao hơn (cần loại bỏ ringing nhiều hơn)
    """
    # Tạo binary edge map
    binary_edges = (edge_map > nguong).astype(float)
    
    # Mở rộng vùng cạnh bằng Gaussian
    weight_map = gaussian_filter(binary_edges, sigma=sigma_expand)
    weight_map = np.clip(weight_map, 0.0, 1.0)
    
    return weight_map


def eicl_mot_kenh(kenh, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5):
    """
    Áp dụng EICL lên một kênh ảnh (Y, hoặc R, G, B)
    
    Args:
        kenh: Kênh ảnh (0-255)
        nguong_canh: Ngưỡng phát hiện cạnh
        sigma_smooth: Độ mạnh làm mượt
        alpha: Cường độ loại bỏ ringing (0-1)
    
    Returns:
        Kênh đã xử lý
    """
    # Phát hiện cạnh
    edge_map = phat_hien_canh_sobel(kenh)
    
    # Tạo bản đồ trọng số
    weight_map = tao_ban_do_trong_so(edge_map, nguong=nguong_canh, sigma_expand=sigma_smooth * 1.5)
    
    # Làm mượt kênh
    kenh_muot = gaussian_filter(kenh, sigma=sigma_smooth)
    
    # Trộn: vùng có ringing (gần cạnh) sẽ được làm mượt nhiều hơn
    kenh_eicl = kenh * (1.0 - alpha * weight_map) + kenh_muot * (alpha * weight_map)
    
    return kenh_eicl, edge_map, weight_map


def eicl_sau_edums(anh_edums, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5, 
                   bgr=False, verbose=False):
    """
    Áp dụng EICL để loại bỏ ringing artifacts sau khi sharpen bằng EDUMS.
    Xử lý trên không gian YCbCr, chỉ làm mượt kênh Y để giữ màu sắc.
    
    Args:
        anh_edums: Ảnh output từ EDUMS
        nguong_canh: Ngưỡng phát hiện cạnh (0.05-0.15 tốt)
        sigma_smooth: Độ mạnh làm mượt (1.0-2.0 tốt)
        alpha: Cường độ loại bỏ ringing (0.3-0.7 tốt)
        bgr: True nếu input là BGR (OpenCV)
        verbose: In thông tin debug
    
    Returns:
        Ảnh đã loại bỏ ringing, cùng dtype/scale với input
    """
    orig_dtype = anh_edums.dtype
    arr = anh_edums.astype(np.float64)
    
    # Kiểm tra scale 0-1
    scaled_from_0_1 = False
    if np.issubdtype(orig_dtype, np.floating) and arr.max() <= 1.0 + 1e-12:
        arr = arr * 255.0
        scaled_from_0_1 = True
    
    # Xử lý alpha channel
    has_alpha = (arr.ndim == 3 and arr.shape[2] == 4)
    alpha_chan = None
    if has_alpha:
        alpha_chan = arr[..., 3].copy()
        arr = arr[..., :3]
    
    if verbose:
        print(f"EICL: shape={arr.shape}, dtype={orig_dtype}, scaled_0_1={scaled_from_0_1}")
    
    # Xử lý ảnh xám
    if arr.ndim == 2:
        Y_eicl, edge_map, weight_map = eicl_mot_kenh(arr, nguong_canh, sigma_smooth, alpha)
        out = np.clip(Y_eicl, 0.0, 255.0)
        
    # Xử lý ảnh màu
    else:
        rgb = arr.copy()
        if bgr:
            rgb = rgb[..., ::-1]  # BGR -> RGB
        
        # Chuyển sang YCbCr
        Y, Cb, Cr = rgb_to_ycbcr_fullrange(rgb)
        
        # Chỉ áp dụng EICL lên kênh Y (giữ nguyên Cb, Cr)
        Y_eicl, edge_map, weight_map = eicl_mot_kenh(Y, nguong_canh, sigma_smooth, alpha)
        Y_eicl = np.clip(Y_eicl, 0.0, 255.0)
        
        # Chuyển ngược về RGB
        out_rgb = ycbcr_to_rgb_fullrange(Y_eicl, Cb, Cr)
        out_rgb = np.clip(out_rgb, 0.0, 255.0)
        
        if bgr:
            out_rgb = out_rgb[..., ::-1]  # RGB -> BGR
        
        out = out_rgb
    
    # Gắn lại alpha channel
    if has_alpha:
        out = np.concatenate([out, alpha_chan[..., np.newaxis]], axis=2)
    
    # Chuyển về dtype/scale gốc
    if scaled_from_0_1:
        out = out / 255.0
        out = out.astype(orig_dtype)
    else:
        if np.issubdtype(orig_dtype, np.integer):
            out = np.clip(out, 0, 255).round().astype(orig_dtype)
        else:
            out = out.astype(orig_dtype)
    
    if verbose:
        print(f"EICL output: shape={out.shape}, dtype={out.dtype}, range=[{out.min():.2f}, {out.max():.2f}]")
    
    return out


In [16]:
img4 = eicl_sau_edums(img3, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5, bgr=True, verbose=True)
cv2.imwrite('15_eicl.png', img4)

EICL: shape=(1600, 2400, 3), dtype=uint8, scaled_0_1=False
EICL output: shape=(1600, 2400, 3), dtype=uint8, range=[0.00, 255.00]


True

In [18]:
# đánh giá chất lượng ảnh

# Use the original upscaled image for comparison
img5 = img1  # Original bicubic upscaled image
img6 = img4  # EIAAF processed image

# Convert img3 (single channel output) to 3-channel for comparison if needed
# Ensure both images have the same shape for comparison
if img5.shape != img6.shape:
	img6 = cv2.cvtColor(img6.astype(np.uint8), cv2.COLOR_GRAY2BGR)

# Tính các chỉ số
psnr_value = psnr(img5, img6)
ssim_value = ssim(img5, img6, channel_axis=-1)
mse_value = mse(img5, img6)
rms_value = np.sqrt(mse_value)

print(f"PSNR: {psnr_value}")
print(f"SSIM: {ssim_value}")
print(f"MSE: {mse_value}")
print(f"RMS: {rms_value}")

PSNR: 44.55354414216843
SSIM: 0.9970841022284304
MSE: 2.278903559027778
RMS: 1.5096037755079237
